# Data Exploration for BAPS ML System

This notebook explores the company data for the Business Analysis & Prediction System.

## Objectives
- Load and understand the dataset
- Perform exploratory data analysis
- Identify patterns and relationships
- Prepare data for feature engineering

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

In [ ]:
# Import ML system modules
import sys
sys.path.append('../src')

from utils.data_loader import DataLoader
from data_processing.validator import DataValidator

## 1. Load Data

In [ ]:
# Initialize data loader
loader = DataLoader("../data")

# Load company data
try:
    df = loader.load_company_data()
    print(f"Successfully loaded data: {df.shape}")
except FileNotFoundError:
    print("No data file found. Creating sample data for demonstration...")
    # Create sample data for demonstration
    np.random.seed(42)
    n_companies = 1000
    
    df = pd.DataFrame({
        'companyName': [f'Company_{i}' for i in range(n_companies)],
        'industry': np.random.choice(['Tech', 'Finance', 'Healthcare', 'Retail', 'Manufacturing'], n_companies),
        'foundedYear': np.random.randint(2000, 2023, n_companies),
        'location': np.random.choice(['US', 'UK', 'Germany', 'France', 'Canada'], n_companies),
        'businessModel': np.random.choice(['B2B', 'B2C', 'Marketplace', 'SaaS'], n_companies),
        'companyStage': np.random.choice(['Seed', 'Series A', 'Series B', 'Series C', 'Mature'], n_companies),
        'revenue': np.random.lognormal(15, 1, n_companies),
        'expenses': np.random.lognormal(14.5, 1, n_companies),
        'profitMargin': np.random.uniform(-0.2, 0.4, n_companies),
        'burnRate': np.random.lognormal(10, 1, n_companies),
        'cashBalance': np.random.lognormal(14, 1.5, n_companies),
        'totalFunding': np.random.lognormal(15, 1.5, n_companies),
        'operationalCost': np.random.lognormal(13, 1, n_companies),
        'marketSize': np.random.lognormal(18, 1, n_companies),
        'competitorCount': np.random.randint(1, 100, n_companies),
        'growthRate': np.random.uniform(-0.1, 2.0, n_companies),
        'marketShare': np.random.uniform(0.01, 20, n_companies),
        'industryGrowthRate': np.random.uniform(0.02, 0.5, n_companies),
        'teamSize': np.random.randint(5, 1000, n_companies),
        'customerCount': np.random.randint(100, 100000, n_companies),
        'churnRate': np.random.uniform(0.01, 0.3, n_companies),
        'nps': np.random.randint(-100, 100, n_companies),
        'customerSatisfaction': np.random.uniform(60, 100, n_companies)
    })
    
    # Save sample data
    loader.save_company_data(df)
    print(f"Created and saved sample data: {df.shape}")

## 2. Basic Data Overview

In [ ]:
# Display basic information
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Display summary statistics
print("Summary Statistics:")
display(df.describe())

## 3. Data Validation

In [ ]:
# Initialize validator
validator = DataValidator()

# Validate data
is_valid, validation_errors = validator.validate_dataframe(df)
print(f"Data Validation: {'PASSED' if is_valid else 'FAILED'}")

if not is_valid:
    print("\nValidation Errors:")
    for row, errors in list(validation_errors.items())[:5]:  # Show first 5 errors
        print(f"{row}: {errors[:3]}")  # Show first 3 errors per row

In [ ]:
# Generate data quality report
quality_report = validator.generate_data_report(df)
print("Data Quality Report:")
print(f"Quality Score: {quality_report['quality_score']:.2f}%")
print(f"Total Records: {quality_report['total_records']}")
print(f"Total Columns: {quality_report['total_columns']}")
print(f"Validation Errors: {len(quality_report['validation_errors'])}")

## 4. Exploratory Data Analysis

### 4.1 Financial Metrics Distribution

In [ ]:
# Financial metrics distribution
financial_cols = ['revenue', 'expenses', 'profitMargin', 'burnRate', 'cashBalance']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(financial_cols):
    if col in df.columns:
        sns.histplot(df[col], kde=True, ax=axes[i])
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel('')

# Remove empty subplot
if len(financial_cols) < len(axes):
    axes[-1].remove()

plt.tight_layout()
plt.show()

### 4.2 Growth and Market Metrics

In [ ]:
# Growth and market metrics
growth_cols = ['growthRate', 'marketShare', 'industryGrowthRate', 'competitorCount']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(growth_cols):
    if col in df.columns:
        sns.histplot(df[col], kde=True, ax=axes[i])
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel('')

plt.tight_layout()
plt.show()

### 4.3 Operational Metrics

In [ ]:
# Operational metrics
operational_cols = ['teamSize', 'customerCount', 'churnRate', 'nps', 'customerSatisfaction']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(operational_cols):
    if col in df.columns:
        sns.histplot(df[col], kde=True, ax=axes[i])
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel('')

# Remove empty subplot
if len(operational_cols) < len(axes):
    axes[-1].remove()

plt.tight_layout()
plt.show()

### 4.4 Categorical Variables Analysis

In [ ]:
# Categorical variables
categorical_cols = ['industry', 'location', 'businessModel', 'companyStage']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    if col in df.columns:
        df[col].value_counts().plot(kind='bar', ax=axes[i])
        axes[i].set_title(f'Distribution of {col}')
        axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Select only numeric columns for correlation
numeric_df = df.select_dtypes(include=[np.number])

# Calculate correlation matrix
correlation_matrix = numeric_df.corr()

# Plot correlation heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Matrix of Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
# Show top correlations with revenue
if 'revenue' in correlation_matrix.columns:
    revenue_correlations = correlation_matrix['revenue'].sort_values(ascending=False)
    print("Top correlations with revenue:")
    print(revenue_correlations.head(10))
    print("\nBottom correlations with revenue:")
    print(revenue_correlations.tail(5))

## 6. Key Relationships

In [ ]:
# Revenue vs Expenses
if 'revenue' in df.columns and 'expenses' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x='revenue', y='expenses', alpha=0.6)
    plt.title('Revenue vs Expenses')
    plt.xlabel('Revenue')
    plt.ylabel('Expenses')
    plt.xscale('log')
    plt.yscale('log')
    plt.show()

In [ ]:
# Growth Rate vs Market Share
if 'growthRate' in df.columns and 'marketShare' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x='marketShare', y='growthRate', alpha=0.6)
    plt.title('Growth Rate vs Market Share')
    plt.xlabel('Market Share (%)')
    plt.ylabel('Growth Rate')
    plt.show()

In [ ]:
# Team Size vs Customer Count
if 'teamSize' in df.columns and 'customerCount' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x='teamSize', y='customerCount', alpha=0.6)
    plt.title('Team Size vs Customer Count')
    plt.xlabel('Team Size')
    plt.ylabel('Customer Count')
    plt.xscale('log')
    plt.yscale('log')
    plt.show()

## 7. Industry-wise Analysis

In [ ]:
# Revenue by industry
if 'industry' in df.columns and 'revenue' in df.columns:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x='industry', y='revenue')
    plt.title('Revenue Distribution by Industry')
    plt.ylabel('Revenue')
    plt.yscale('log')
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
# Growth rate by industry
if 'industry' in df.columns and 'growthRate' in df.columns:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x='industry', y='growthRate')
    plt.title('Growth Rate Distribution by Industry')
    plt.ylabel('Growth Rate')
    plt.xticks(rotation=45)
    plt.show()

## 8. Summary and Next Steps

In [ ]:
# Summary statistics
print("=== DATA EXPLORATION SUMMARY ===")
print(f"Dataset Shape: {df.shape}")
print(f"Numeric Columns: {len(df.select_dtypes(include=[np.number]).columns)}")
print(f"Categorical Columns: {len(df.select_dtypes(include=['object']).columns)}")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Data Quality Score: {quality_report['quality_score']:.2f}%")

if 'revenue' in df.columns:
    print(f"\nRevenue Statistics:")
    print(f"  Mean: ${df['revenue'].mean():,.0f}")
    print(f"  Median: ${df['revenue'].median():,.0f}")
    print(f"  Std: ${df['revenue'].std():,.0f}")

if 'growthRate' in df.columns:
    print(f"\nGrowth Rate Statistics:")
    print(f"  Mean: {df['growthRate'].mean():.2f}")
    print(f"  Median: {df['growthRate'].median():.2f}")
    print(f"  Std: {df['growthRate'].std():.2f}")

### Key Findings:

1. **Data Quality**: The dataset has [quality_score]% data quality score
2. **Revenue Distribution**: [description of revenue distribution]
3. **Growth Patterns**: [description of growth patterns]
4. **Industry Differences**: [description of industry differences]
5. **Correlations**: [key correlations found]

### Next Steps:

1. **Data Cleaning**: Handle missing values and outliers
2. **Feature Engineering**: Create new features based on insights
3. **Preprocessing**: Scale and encode features
4. **Model Training**: Train ML models
5. **Evaluation**: Evaluate model performance

Proceed to the next notebook: `02_feature_engineering.ipynb`